# 2026 March Madness Bracket Predictions

Use this notebook to explore 2026 NCAA Men's Tournament predictions from the latest model output.

The model uses ELO, heat momentum, team form, box score features, schedule strength, and lines context. Probabilities are calibrated and clipped in the production pipeline.

This notebook shows:
1. Team power rankings for seeded teams
2. Round-of-64 matchup probabilities by region
3. Cinderella watch using heat features
4. Head-to-head matchup explorer
5. Submission readiness checks

---

In [15]:
from __future__ import annotations
import sys
from pathlib import Path

import polars as pl

ROOT = Path("..").resolve()
sys.path.insert(0, str(ROOT / "src"))
BASE = ROOT / "artifacts" / "latest"

# Load submission predictions
submission = pl.read_csv(BASE / "submission.csv")

# Load team metadata
teams = pl.read_csv(ROOT / "data" / "MTeams.csv").select("TeamID", "TeamName")
tid_to_name = dict(zip(teams["TeamID"].to_list(), teams["TeamName"].to_list()))

# Load 2026 seeds and slots
seeds_2026 = pl.read_csv(ROOT / "data" / "MNCAATourneySeeds.csv").filter(pl.col("Season") == 2026)
slots_2026 = pl.read_csv(ROOT / "data" / "MNCAATourneySlots.csv").filter(pl.col("Season") == 2026)

# Load team features for additional context
team_features = pl.read_parquet(BASE / "gold" / "team_season_features.parquet")
m_2026 = team_features.filter((pl.col("sex") == "M") & (pl.col("season") == 2026))

# Build seed -> team mapping
seed_to_team = {}
seed_to_name = {}
for row in seeds_2026.iter_rows(named=True):
    tid = row["TeamID"]
    seed_to_team[row["Seed"]] = tid
    seed_to_name[row["Seed"]] = tid_to_name.get(tid, str(tid))


def get_pred(tid_a: int, tid_b: int) -> float:
    """Get P(lower-ID team wins) from submission."""
    low, high = min(tid_a, tid_b), max(tid_a, tid_b)
    match = submission.filter(pl.col("ID") == f"2026_{low}_{high}")
    if match.height > 0:
        return float(match["Pred"][0])
    return 0.5


REGION_NAMES = {"W": "West", "X": "East", "Y": "South", "Z": "Midwest"}

print(f"Loaded {seeds_2026.height} seeded teams across 4 regions")
print(f"Loaded {slots_2026.height} bracket slots")
print(f"Submission has {submission.height:,} predictions")

Loaded 68 seeded teams across 4 regions
Loaded 67 bracket slots
Submission has 132,133 predictions


---

## Team Power Rankings

All 68 tournament teams ranked by season-end ELO rating. Higher ELO = stronger team based on regular season performance. Heat scores show recent momentum (positive = outperforming expectations).

In [16]:
# Power rankings: all 68 teams sorted by ELO
power = (
    m_2026
    .join(seeds_2026.select(
        pl.col("TeamID").cast(pl.Int64).alias("team_id"),
        pl.col("Seed"),
    ), on="team_id", how="inner")
    .join(teams.rename({"TeamID": "team_id"}), on="team_id", how="left")
    .sort("season_end_elo", descending=True)
)

print(f"{'#':>3} {'Seed':<6} {'Team':<22} {'ELO':>7} {'Record':>8} {'WR':>6} {'Margin':>7} {'SOS':>7} {'H-5g':>7} {'FG%':>6}")
print("─" * 95)
for i, row in enumerate(power.iter_rows(named=True), 1):
    record = f"{row['wins']}-{row['losses']}"
    h5 = row.get('pre_tourney_heat_5g') or 0
    elo = row.get('season_end_elo') or 1500
    sos = row.get('sos') or 0
    fg = row.get('fg_pct') or 0
    print(f"{i:>3} {row['Seed']:<6} {row['TeamName']:<22} {elo:>7.0f} {record:>8} {row['win_rate']:>5.1%} {row['avg_margin']:>+7.1f} {sos:>7.0f} {h5:>+7.1f} {fg:>5.1%}")

  # Seed   Team                       ELO   Record     WR  Margin     SOS    H-5g    FG%
───────────────────────────────────────────────────────────────────────────────────────────────
  1 W01    Duke                      1982     32-2 94.1%   +19.1    1616    +1.3 49.0%
  2 Z01    Arizona                   1948     32-2 94.1%   +17.4    1613    +0.7 50.2%
  3 Y01    Michigan                  1944     31-3 91.2%   +17.6    1637    -4.1 50.5%
  4 W05    St John's                 1883     28-6 82.4%   +11.6    1575    -1.6 45.4%
  5 X01    Florida                   1881     26-7 78.8%   +14.8    1620    +6.0 47.8%
  6 Z03    Gonzaga                   1869     30-3 90.9%   +19.1    1540    -1.1 51.0%
  7 X02    Houston                   1844     28-6 82.4%   +14.3    1600    +5.0 45.0%
  8 Y03    Virginia                  1838     29-5 85.3%   +12.2    1567    -5.4 46.3%
  9 Z02    Purdue                    1836     27-8 77.1%   +11.5    1621    -0.6 49.9%
 10 X07    St Mary's CA         

---

## Round of 64 — First Round Matchups by Region

Each region has 8 first-round games (1v16, 2v15, 3v14, ... 8v9). The model predicts the probability the higher-seeded team (stronger seed = lower number) wins.

**⚠️ Upset Alert** = model gives the lower-seeded team (underdog) a >35% chance of winning.

In [17]:
# First round matchups by region
first_round_slots = slots_2026.filter(pl.col("Slot").str.starts_with("R1")).sort("Slot")

for region_code, region_name in REGION_NAMES.items():
    region_slots = first_round_slots.filter(
        pl.col("Slot").str.contains(f"R1{region_code}")
    ).sort("Slot")

    print(f"\n{'=' * 78}")
    print(f"  {region_name.upper()} REGION ({region_code})")
    print(f"{'=' * 78}")
    print(f"{'Slot':<7} {'Strong':<22} {'vs':^4} {'Weak':<22} {'P(Strong)':>10} {'Pick'}")
    print("─" * 78)

    for row in region_slots.iter_rows(named=True):
        strong_seed = row["StrongSeed"]
        weak_seed = row["WeakSeed"]
        strong_tid = seed_to_team.get(strong_seed, 0)
        weak_tid = seed_to_team.get(weak_seed, 0)
        strong_name = f"({strong_seed[-2:]}) {seed_to_name.get(strong_seed, '?')}"
        weak_name = f"({weak_seed[-2:]}) {seed_to_name.get(weak_seed, '?')}"

        if strong_tid and weak_tid:
            # Get probability that the strong seed wins
            low_id = min(strong_tid, weak_tid)
            high_id = max(strong_tid, weak_tid)
            p_low_wins = get_pred(strong_tid, weak_tid)
            # If strong_tid is the low_id, p_strong = p_low_wins
            # If strong_tid is the high_id, p_strong = 1 - p_low_wins
            p_strong = p_low_wins if strong_tid == low_id else 1 - p_low_wins

            upset = "⚠️ UPSET ALERT" if p_strong < 0.65 else ""
            winner = strong_name if p_strong >= 0.5 else weak_name
            print(f"{row['Slot']:<7} {strong_name:<22} {'vs':^4} {weak_name:<22} {p_strong:>9.1%} {winner}  {upset}")
        else:
            print(f"{row['Slot']:<7} {strong_seed:<22} {'vs':^4} {weak_seed:<22} {'N/A':>10}")


  WEST REGION (W)
Slot    Strong                  vs  Weak                    P(Strong) Pick
──────────────────────────────────────────────────────────────────────────────
R1W1    (01) Duke               vs  (16) Siena                 99.0% (01) Duke  
R1W2    (02) Connecticut        vs  (15) Furman                99.0% (02) Connecticut  
R1W3    (03) Michigan St        vs  (14) N Dakota St           78.1% (03) Michigan St  
R1W4    (04) Kansas             vs  (13) Cal Baptist           75.0% (04) Kansas  
R1W5    (05) St John's          vs  (12) Northern Iowa         85.7% (05) St John's  
R1W6    (06) Louisville         vs  (11) South Florida         75.0% (06) Louisville  
R1W7    (07) UCLA               vs  (10) UCF                   71.4% (07) UCLA  
R1W8    (08) Ohio St            vs  (09) TCU                   59.4% (08) Ohio St  ⚠️ UPSET ALERT

  EAST REGION (X)
Slot    Strong                  vs  Weak                    P(Strong) Pick
─────────────────────────────────────────

---

## Cinderella Watch 🔮

These are the teams with the **highest pre-tournament heat scores** — teams that have been outperforming their ELO expectations heading into March. The Cinderella System thesis says these teams are most likely to pull upsets.

"Heat" measures per-game over-performance (actual margin minus ELO-expected margin), averaged over the last 5 games. The ELO engine now uses **margin-of-victory scaling** and **home court adjustment**, making the heat signal more accurate than a simple win/loss ELO.

In [18]:
import re

# Cinderella Watch — hottest lower seeds entering the tournament
cinderella_candidates = (
    power
    .with_columns(
        pl.col("Seed").str.extract(r"(\d+)", 1).cast(pl.Int64).alias("seed_num_col"),
    )
    .filter(pl.col("seed_num_col") >= 7)  # Focus on teams seeded 7 or worse (upset candidates)
    .sort("pre_tourney_heat_5g", descending=True)
)

print("🔮 CINDERELLA CANDIDATES — Lower Seeds with Highest Heat\n")
print(f"{'Seed':<6} {'Team':<22} {'ELO':>7} {'Record':>8} {'H-5g':>7} {'H-3g':>7} {'H-1g':>7} {'Margin':>7}")
print("─" * 80)

for row in cinderella_candidates.head(15).iter_rows(named=True):
    h1 = row['pre_tourney_heat_1g'] or 0
    h3 = row['pre_tourney_heat_3g'] or 0
    h5 = row['pre_tourney_heat_5g'] or 0
    record = f"{row['wins']}-{row['losses']}"
    elo = row['season_end_elo'] or 1500
    print(f"{row['Seed']:<6} {row['TeamName']:<22} {elo:>7.0f} {record:>8} {h5:>+7.1f} {h3:>+7.1f} {h1:>+7.1f} {row['avg_margin']:>+7.1f}")

# Show their first-round matchup predictions
print(f"\n{'─' * 80}")
print(f"\nFirst-Round Upset Probabilities for Top Cinderella Candidates:\n")

for row in cinderella_candidates.head(10).iter_rows(named=True):
    seed = row["Seed"]
    region = seed[0]
    sn = int(re.search(r"\d+", seed).group())
    opp_sn = 17 - sn  # e.g., 12 seed plays 5 seed
    opp_seed = f"{region}{opp_sn:02d}"

    tid = row["team_id"]
    opp_tid = seed_to_team.get(opp_seed, 0)
    opp_name = seed_to_name.get(opp_seed, "?")

    if opp_tid:
        low_id = min(tid, opp_tid)
        high_id = max(tid, opp_tid)
        p_low = get_pred(tid, opp_tid)
        # P(this team wins): depends on whether this team is low or high
        p_win = p_low if tid == low_id else 1 - p_low
        icon = "🔥" if p_win > 0.35 else "  "
        print(f"  {icon} ({sn:>2}) {row['TeamName']:<20} vs ({opp_sn:>2}) {opp_name:<20}  P(upset) = {p_win:.1%}")

🔮 CINDERELLA CANDIDATES — Lower Seeds with Highest Heat

Seed   Team                       ELO   Record    H-5g    H-3g    H-1g  Margin
────────────────────────────────────────────────────────────────────────────────
X16b   Prairie View              1472    14-17   +14.4   +12.0   +20.7    -2.8
Y15    Tennessee St              1667     20-9   +14.1   +17.3    +6.0    +4.1
Y16b   UMBC                      1647     22-8    +8.7   +10.2    +5.0    +7.5
Y16a   Howard                    1600    19-10    +8.5    +2.3    +1.7    +7.2
W07    UCLA                      1716    23-11    +8.1    +8.9   +10.2    +6.7
X15    Idaho                     1556    18-14    +7.9   +11.8   +13.2    +2.7
W09    TCU                       1690    22-11    +5.9    +8.5    -1.5    +6.2
X14    Penn                      1612    17-11    +5.6    +6.5    +4.4    +1.2
W08    Ohio St                   1702    21-12    +5.2    +7.9    +1.2    +7.0
Y08    Georgia                   1676    22-10    +4.8    +4.3    -0.6  

---

## Matchup Explorer

Use the function below to compare any two teams head-to-head. Pass team names (partial match) or TeamIDs.

In [19]:
def compare(team_a, team_b):
    """Compare two teams head-to-head. Pass names (partial match) or integer TeamIDs."""

    def resolve(team):
        if isinstance(team, int):
            return team, tid_to_name.get(team, str(team))
        # Partial name match
        matches = teams.filter(pl.col("TeamName").str.to_lowercase().str.contains(str(team).lower()))
        if matches.height == 0:
            print(f"No team found matching '{team}'")
            return None, None
        if matches.height > 1:
            print(f"Multiple matches for '{team}': {matches['TeamName'].to_list()}")
        tid = int(matches["TeamID"][0])
        return tid, matches["TeamName"][0]

    tid_a, name_a = resolve(team_a)
    tid_b, name_b = resolve(team_b)
    if tid_a is None or tid_b is None:
        return

    low_id, high_id = min(tid_a, tid_b), max(tid_a, tid_b)
    low_name = name_a if tid_a == low_id else name_b
    high_name = name_b if tid_a == low_id else name_a

    # Get features for both teams
    feat_a = m_2026.filter(pl.col("team_id") == tid_a)
    feat_b = m_2026.filter(pl.col("team_id") == tid_b)

    if feat_a.height == 0 or feat_b.height == 0:
        print("One or both teams not found in 2026 M features")
        return

    fa = feat_a.to_dicts()[0]
    fb = feat_b.to_dicts()[0]

    # Get seeds if available
    seed_a = seeds_2026.filter(pl.col("TeamID") == tid_a)
    seed_b = seeds_2026.filter(pl.col("TeamID") == tid_b)
    seed_a_str = seed_a["Seed"][0] if seed_a.height > 0 else "N/A"
    seed_b_str = seed_b["Seed"][0] if seed_b.height > 0 else "N/A"

    p_low = get_pred(tid_a, tid_b)
    p_a = p_low if tid_a == low_id else 1 - p_low

    print(f"{'═' * 70}")
    print(f"  {name_a} ({seed_a_str}) vs {name_b} ({seed_b_str})")
    print(f"{'═' * 70}")
    print(f"\n  P({name_a} wins) = {p_a:.1%}    |    P({name_b} wins) = {1 - p_a:.1%}\n")

    stats = [
        ("Record", f"{fa['wins']}-{fa['losses']}", f"{fb['wins']}-{fb['losses']}"),
        ("Win Rate", f"{fa['win_rate']:.1%}", f"{fb['win_rate']:.1%}"),
        ("Avg Margin", f"{fa['avg_margin']:+.1f}", f"{fb['avg_margin']:+.1f}"),
        ("Avg Pts For", f"{fa['avg_pts_for']:.1f}", f"{fb['avg_pts_for']:.1f}"),
        ("Avg Pts Against", f"{fa['avg_pts_against']:.1f}", f"{fb['avg_pts_against']:.1f}"),
        ("Last 5 Win Rate", f"{fa['last5_win_rate']:.1%}", f"{fb['last5_win_rate']:.1%}"),
        ("Last 5 Margin", f"{fa['last5_avg_margin']:+.1f}", f"{fb['last5_avg_margin']:+.1f}"),
        ("ELO", f"{(fa.get('season_end_elo') or 1500):.0f}", f"{(fb.get('season_end_elo') or 1500):.0f}"),
        ("SOS", f"{(fa.get('sos') or 0):.0f}", f"{(fb.get('sos') or 0):.0f}"),
        ("FG%", f"{(fa.get('fg_pct') or 0):.1%}", f"{(fb.get('fg_pct') or 0):.1%}"),
        ("3PT%", f"{(fa.get('fg3_pct') or 0):.1%}", f"{(fb.get('fg3_pct') or 0):.1%}"),
        ("FT%", f"{(fa.get('ft_pct') or 0):.1%}", f"{(fb.get('ft_pct') or 0):.1%}"),
        ("Reb Margin", f"{(fa.get('avg_reb_margin') or 0):+.1f}", f"{(fb.get('avg_reb_margin') or 0):+.1f}"),
        ("Ast/TO", f"{(fa.get('ast_to_ratio') or 0):.2f}", f"{(fb.get('ast_to_ratio') or 0):.2f}"),
        ("Heat (1g)", f"{(fa.get('pre_tourney_heat_1g') or 0):+.1f}", f"{(fb.get('pre_tourney_heat_1g') or 0):+.1f}"),
        ("Heat (3g)", f"{(fa.get('pre_tourney_heat_3g') or 0):+.1f}", f"{(fb.get('pre_tourney_heat_3g') or 0):+.1f}"),
        ("Heat (5g)", f"{(fa.get('pre_tourney_heat_5g') or 0):+.1f}", f"{(fb.get('pre_tourney_heat_5g') or 0):+.1f}"),
    ]

    print(f"  {'Stat':<18} {name_a:>18} {name_b:>18}")
    print(f"  {'─' * 56}")
    for label, val_a, val_b in stats:
        print(f"  {label:<18} {val_a:>18} {val_b:>18}")

    print(f"\n  PICK: {'→ ' + name_a if p_a >= 0.5 else '→ ' + name_b} ({max(p_a, 1 - p_a):.1%})")


# Example: compare two teams
compare("Duke", "Auburn")

══════════════════════════════════════════════════════════════════════
  Duke (W01) vs Auburn (N/A)
══════════════════════════════════════════════════════════════════════

  P(Duke wins) = 85.7%    |    P(Auburn wins) = 14.3%

  Stat                             Duke             Auburn
  ────────────────────────────────────────────────────────
  Record                           32-2              17-16
  Win Rate                        94.1%              51.5%
  Avg Margin                      +19.1               +3.3
  Avg Pts For                      82.3               82.7
  Avg Pts Against                  63.1               79.4
  Last 5 Win Rate                100.0%              40.0%
  Last 5 Margin                   +12.2               +0.8
  ELO                              1982               1582
  SOS                              1616               1649
  FG%                             49.0%              45.7%
  3PT%                            35.1%              33.8%
  FT% 

In [20]:
# Try your own matchups! Uncomment or edit these:
# compare("Houston", "Florida")
# compare("Kansas", "Michigan St")
# compare(1385, 1120)  # St John's vs Auburn (by TeamID)
compare("Connecticut", "St John")

══════════════════════════════════════════════════════════════════════
  Connecticut (W02) vs St John's (W05)
══════════════════════════════════════════════════════════════════════

  P(Connecticut wins) = 73.7%    |    P(St John's wins) = 26.3%

  Stat                      Connecticut          St John's
  ────────────────────────────────────────────────────────
  Record                           29-5               28-6
  Win Rate                        85.3%              82.4%
  Avg Margin                      +12.4              +11.6
  Avg Pts For                      77.5               81.6
  Avg Pts Against                  65.1               70.0
  Last 5 Win Rate                 60.0%             100.0%
  Last 5 Margin                    +3.8              +10.6
  ELO                              1813               1883
  SOS                              1583               1575
  FG%                             48.2%              45.4%
  3PT%                            35.2%      

---

## Monte Carlo Tournament Simulation (100K Runs, Heat-Adjusted)

This simulation runs 100,000 independent bracket simulations with a **momentum / heat adjustment**: when a team wins a game where they were the underdog, they receive a probability boost in their next matchup. The idea is that March Madness "hot" teams don't follow static probabilities — a 12-seed that beats a 5-seed is legitimately more dangerous in the next round.

**Heat Boost Mechanics:**
- After each game, if Team A wins and the model gave them less than a 50% chance, they receive an **upset heat boost** proportional to how big the upset was
- The boost is applied as a logit-space shift (so it compresses properly near 0/1 boundaries)
- Boosts decay slightly each round (a team stays hot but doesn't compound forever)
- This makes Cinderella runs more likely than a static-probability model would predict

**Three Bracket Strategies:**
1. **Modal Bracket** — pick the winner of each slot who wins the *most simulations* (best single bracket for expected score)
2. **Chalk+ Bracket** — favor higher seeds except where the model sees a clear upset (>40% underdog)
3. **Upset Special** — use simulation advancement rates but pick 2-3 "value upsets" per region where heat-adjusted probability diverges from seed expectation

In [ ]:
import numpy as np
import time

# ──────────────────────────────────────────────────────────────────────
#  Build probability lookup from submission
# ──────────────────────────────────────────────────────────────────────

prob_lookup: dict[tuple[int, int], float] = {}
for row in submission.iter_rows(named=True):
    parts = row["ID"].split("_")
    prob_lookup[(int(parts[1]), int(parts[2]))] = float(row["Pred"])

# Get pre-tournament heat scores per team for initial heat state
team_heat_init: dict[int, float] = {}
for row in m_2026.iter_rows(named=True):
    tid = int(row["team_id"])
    h5 = row.get("pre_tourney_heat_5g") or 0.0
    team_heat_init[tid] = float(h5)


def get_prob(tid_a: int, tid_b: int, heat_a: float, heat_b: float) -> float:
    """Get win probability for tid_a vs tid_b, adjusted by accumulated heat.

    Works in logit space so boosts near 0/1 boundaries stay well-calibrated.
    """
    low, high = min(tid_a, tid_b), max(tid_a, tid_b)
    base_prob = prob_lookup.get((low, high), 0.5)

    # Convert to P(tid_a wins)
    p_a = base_prob if tid_a == low else 1.0 - base_prob

    # Apply heat differential in logit space
    heat_diff = heat_a - heat_b
    if abs(heat_diff) < 1e-6:
        return p_a

    # Logit transform: log(p / (1-p))
    p_a = np.clip(p_a, 0.005, 0.995)
    logit_p = np.log(p_a / (1.0 - p_a))

    # Heat boost magnitude: scale factor controls how much heat matters
    # 0.03 per heat point ≈ a team with +10 heat gets ~0.3 logit boost (~7% at p=0.5)
    HEAT_SCALE = 0.03
    logit_p += heat_diff * HEAT_SCALE

    return float(1.0 / (1.0 + np.exp(-logit_p)))


# ──────────────────────────────────────────────────────────────────────
#  Monte Carlo Simulation with Heat Adjustment
# ──────────────────────────────────────────────────────────────────────

N_SIMS = 100_000
UPSET_HEAT_BOOST = 4.0    # Heat points gained when winning as underdog
EXPECTED_WIN_BOOST = 0.5   # Small heat gain for winning as favorite
HEAT_DECAY = 0.85          # Heat retention between rounds (prevents runaway)

# Parse bracket structure
slots = sorted(
    [{"slot": r["Slot"], "strong": r["StrongSeed"], "weak": r["WeakSeed"]}
     for r in slots_2026.iter_rows(named=True)],
    key=lambda s: s["slot"],
)

# Map all keys (seeds + slots) to array indices
all_keys: set[str] = set(seed_to_team.keys())
for s in slots:
    all_keys.update([s["slot"], s["strong"], s["weak"]])
key_to_idx = {k: i for i, k in enumerate(sorted(all_keys))}
n_keys = len(key_to_idx)

# Track how many times each team reaches each round
ROUND_NAMES = {0: "R64", 1: "R32", 2: "S16", 3: "E8", 4: "F4", 5: "NCG", 6: "Champ"}
team_round_counts: dict[int, dict[int, int]] = {}  # {tid: {round_idx: count}}
champion_counts: dict[int, int] = {}
slot_winner_counts: dict[str, dict[int, int]] = {}  # {slot_name: {tid: count}}

# All seeded teams start in round 0
for tid in seed_to_team.values():
    team_round_counts[tid] = {0: N_SIMS}  # everyone plays round of 64

print(f"Running {N_SIMS:,} bracket simulations with heat adjustment...")
t0 = time.time()

rng = np.random.default_rng(42)

# Pre-allocate arrays
team_at = np.zeros((N_SIMS, n_keys), dtype=np.int32)
heat_at = np.zeros((N_SIMS, n_keys), dtype=np.float32)

# Initialize seed positions and starting heat
for seed_str, tid in seed_to_team.items():
    idx = key_to_idx[seed_str]
    team_at[:, idx] = tid
    heat_at[:, idx] = team_heat_init.get(tid, 0.0)

# Pre-generate all random draws
draws = rng.random((N_SIMS, len(slots)), dtype=np.float32)

for game_idx, slot_info in enumerate(slots):
    si = key_to_idx[slot_info["strong"]]
    wi = key_to_idx[slot_info["weak"]]
    oi = key_to_idx[slot_info["slot"]]

    ta = team_at[:, si]  # team IDs on strong side
    tb = team_at[:, wi]  # team IDs on weak side
    ha = heat_at[:, si]  # heat on strong side
    hb = heat_at[:, wi]  # heat on weak side

    # Determine round from slot name (R1->round 1, R2->round 2, etc.)
    round_num = int(slot_info["slot"][1]) if slot_info["slot"][1].isdigit() else 1

    # Find unique matchups across all sims
    t_low = np.minimum(ta, tb)
    t_high = np.maximum(ta, tb)
    unique_matchups = np.unique(np.column_stack([t_low, t_high]), axis=0)

    for matchup in unique_matchups:
        ml, mh = int(matchup[0]), int(matchup[1])
        if ml == 0 or mh == 0:
            continue

        mask = (t_low == ml) & (t_high == mh)
        count = int(mask.sum())
        if count == 0:
            continue

        # Get per-sim heat for this matchup
        # "a" is always the team on the strong side, "b" on the weak side
        # We need to figure out which of (ml, mh) is on which side
        heat_strong = ha[mask]
        heat_weak = hb[mask]

        # Map to (ml, mh) orientation
        strong_is_low = ta[mask] == ml
        heat_low = np.where(strong_is_low, heat_strong, heat_weak)
        heat_high = np.where(strong_is_low, heat_weak, heat_strong)

        # Compute heat-adjusted probability for each sim
        base_p = prob_lookup.get((ml, mh), 0.5)
        base_p_clipped = np.clip(base_p, 0.005, 0.995)
        base_logit = np.log(base_p_clipped / (1.0 - base_p_clipped))

        HEAT_SCALE = 0.03
        heat_diff = heat_low - heat_high
        adj_logit = base_logit + heat_diff * HEAT_SCALE
        adj_prob = 1.0 / (1.0 + np.exp(-adj_logit))

        # Determine winners
        low_wins = draws[mask, game_idx] < adj_prob

        # Track advancement
        slot_name = slot_info["slot"]
        if slot_name not in slot_winner_counts:
            slot_winner_counts[slot_name] = {}

        n_low_wins = int(low_wins.sum())
        n_high_wins = count - n_low_wins

        if n_low_wins > 0:
            slot_winner_counts[slot_name][ml] = slot_winner_counts[slot_name].get(ml, 0) + n_low_wins
            if ml not in team_round_counts:
                team_round_counts[ml] = {}
            team_round_counts[ml][round_num] = team_round_counts[ml].get(round_num, 0) + n_low_wins

        if n_high_wins > 0:
            slot_winner_counts[slot_name][mh] = slot_winner_counts[slot_name].get(mh, 0) + n_high_wins
            if mh not in team_round_counts:
                team_round_counts[mh] = {}
            team_round_counts[mh][round_num] = team_round_counts[mh].get(round_num, 0) + n_high_wins

        # Championship tracking
        if slot_name == "R6CH":
            champion_counts[ml] = champion_counts.get(ml, 0) + n_low_wins
            champion_counts[mh] = champion_counts.get(mh, 0) + n_high_wins

        # Update winner into the output slot
        # For sims in this matchup mask, place the winning team
        winner_ids = np.where(low_wins, ml, mh)
        loser_ids = np.where(low_wins, mh, ml)

        # Compute heat update: upset winners get big boost, expected winners get small boost
        winner_was_underdog = np.where(low_wins, base_p < 0.5, base_p > 0.5)
        upset_magnitude = np.where(low_wins,
                                    np.abs(0.5 - base_p),
                                    np.abs(0.5 - (1.0 - base_p)))

        heat_boost = np.where(winner_was_underdog,
                              UPSET_HEAT_BOOST * (1.0 + upset_magnitude),
                              EXPECTED_WIN_BOOST)

        # Winner's heat = decayed previous heat + new boost
        winner_heat_low = heat_low * HEAT_DECAY + np.where(low_wins, heat_boost, 0.0)
        winner_heat_high = heat_high * HEAT_DECAY + np.where(~low_wins, heat_boost, 0.0)
        new_heat = np.where(low_wins, winner_heat_low, winner_heat_high).astype(np.float32)

        # Write back to arrays using advanced indexing
        sim_indices = np.where(mask)[0]
        team_at[sim_indices, oi] = winner_ids
        heat_at[sim_indices, oi] = new_heat

elapsed = time.time() - t0
print(f"Done in {elapsed:.1f}s — {N_SIMS / elapsed:,.0f} sims/sec")
print(f"\nChampionship contenders (top 15):")
print(f"{'Team':<25} {'Seed':<6} {'Champ%':>8} {'F4%':>8} {'E8%':>8} {'S16%':>8}")
print("─" * 65)

# Reverse lookup: tid -> seed string
tid_to_seed = {v: k for k, v in seed_to_team.items()}

champ_sorted = sorted(champion_counts.items(), key=lambda x: -x[1])
for tid, cnt in champ_sorted[:15]:
    name = tid_to_name.get(tid, str(tid))
    seed = tid_to_seed.get(tid, "?")
    champ_pct = cnt / N_SIMS * 100
    rc = team_round_counts.get(tid, {})
    f4_pct = rc.get(5, 0) / N_SIMS * 100   # R5 = Final Four
    e8_pct = rc.get(4, 0) / N_SIMS * 100   # R4 = Elite Eight
    s16_pct = rc.get(3, 0) / N_SIMS * 100  # R3 = Sweet 16
    print(f"{name:<25} {seed:<6} {champ_pct:>7.1f}% {f4_pct:>7.1f}% {e8_pct:>7.1f}% {s16_pct:>7.1f}%")

In [ ]:
# Debug: check slot names, champion_counts, and slot_winner_counts
print("All slot names:", [s["slot"] for s in slots])
print("\nchampion_counts:", champion_counts)
print("\nslots with winners:", {k: len(v) for k, v in slot_winner_counts.items()})
print("\nLast 5 slots:")
for s in slots[-5:]:
    print(f"  {s}")
print("\nTotal teams tracked:", len(team_round_counts))
print("Sample team_round_counts:", dict(list(team_round_counts.items())[:3]))

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick

# ──────────────────────────────────────────────────────────────────────
#  Cinderella Runs — teams seeded 10+ that make deep runs
# ──────────────────────────────────────────────────────────────────────

cinderella_data = []
for tid, rc in team_round_counts.items():
    seed_str = tid_to_seed.get(tid, "?")
    seed_num = int(seed_str[1:3]) if seed_str != "?" else 0
    if seed_num >= 10:
        s16_pct = rc.get(3, 0) / N_SIMS * 100
        e8_pct = rc.get(4, 0) / N_SIMS * 100
        if s16_pct > 1.0:  # At least 1% chance of S16
            cinderella_data.append({
                "name": tid_to_name.get(tid, str(tid)),
                "seed": seed_str,
                "seed_num": seed_num,
                "tid": tid,
                "R32": rc.get(1, 0) / N_SIMS * 100,
                "S16": s16_pct,
                "E8": e8_pct,
                "F4": rc.get(5, 0) / N_SIMS * 100,
            })

cinderella_data.sort(key=lambda x: -x["S16"])

print("🏀 Cinderella Watch — Seeds 10+ with Sweet 16 Potential\n")
print(f"{'Team':<25} {'Seed':<6} {'R32%':>7} {'S16%':>7} {'E8%':>7} {'F4%':>7}")
print("─" * 60)
for c in cinderella_data[:12]:
    print(f"{c['name']:<25} {c['seed']:<6} {c['R32']:>6.1f}% {c['S16']:>6.1f}% {c['E8']:>6.1f}% {c['F4']:>6.1f}%")

# ──────────────────────────────────────────────────────────────────────
#  Advancement Probability Chart — top 12 teams
# ──────────────────────────────────────────────────────────────────────

top_12_tids = [tid for tid, _ in champ_sorted[:12]]
rounds_to_plot = [1, 2, 3, 4, 5, 6]  # R32 through Champ

fig, ax = plt.subplots(figsize=(14, 7))
x = np.arange(len(rounds_to_plot))
width = 0.06
n_teams = len(top_12_tids)

colors = plt.cm.tab20(np.linspace(0, 1, n_teams))

for i, tid in enumerate(top_12_tids):
    rc = team_round_counts.get(tid, {})
    vals = [rc.get(r, 0) / N_SIMS * 100 for r in rounds_to_plot]
    offset = (i - n_teams / 2) * width
    bars = ax.bar(x + offset, vals, width, label=tid_to_name.get(tid, str(tid)),
                  color=colors[i], alpha=0.85)

ax.set_xticks(x)
ax.set_xticklabels(["R32", "S16", "E8", "F4", "NCG", "Champ"])
ax.set_ylabel("Advancement Probability (%)")
ax.set_title(f"Tournament Advancement — Top 12 Contenders ({N_SIMS:,} sims, heat-adjusted)")
ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=8)
ax.yaxis.set_major_formatter(mtick.PercentFormatter())
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

## 🏆 Bracket Selection Strategies

Three approaches to turn simulation results into a printable bracket:

| Strategy | Philosophy | Best For |
|----------|-----------|----------|
| **Modal Bracket** | Pick the most frequent winner of each slot across all sims | Maximizing expected correct picks — your "safest" bracket |
| **Chalk+ Bracket** | Favor expected seeds, but override when the model gives underdogs >40% | Office pools where you need a few upsets but can't go crazy |
| **Upset Special** | Find games where heat-adjusted win% diverges most from seed expectation, and lean into those upsets | Pools with many entries where you need differentiation |

In [ ]:
# ──────────────────────────────────────────────────────────────────────
#  Strategy 1: Modal Bracket — most frequent winner of each slot
# ──────────────────────────────────────────────────────────────────────

def build_modal_bracket() -> dict[str, tuple[int, str, float]]:
    """Returns {slot: (tid, team_name, win_pct)} picking the most frequent winner."""
    bracket = {}
    for slot_info in slots:
        slot_name = slot_info["slot"]
        winners = slot_winner_counts.get(slot_name, {})
        if not winners:
            continue
        best_tid = max(winners, key=lambda t: winners[t])
        pct = winners[best_tid] / N_SIMS * 100
        bracket[slot_name] = (best_tid, tid_to_name.get(best_tid, str(best_tid)), pct)
    return bracket


# ──────────────────────────────────────────────────────────────────────
#  Strategy 2: Chalk+ — favor seeds, override on strong upsets (>40%)
# ──────────────────────────────────────────────────────────────────────

def build_chalk_plus_bracket() -> dict[str, tuple[int, str, float]]:
    """Favor the higher-seeded team unless the lower seed wins >40% of sims."""
    bracket = {}
    advancing: dict[str, int] = dict(seed_to_team)  # seed -> tid

    for slot_info in slots:
        slot_name = slot_info["slot"]
        strong_key = slot_info["strong"]
        weak_key = slot_info["weak"]

        tid_strong = advancing.get(strong_key)
        tid_weak = advancing.get(weak_key)
        if tid_strong is None or tid_weak is None:
            continue

        winners = slot_winner_counts.get(slot_name, {})
        total = sum(winners.values()) or 1
        weak_pct = winners.get(tid_weak, 0) / total

        # In "Chalk+" we pick the strong seed UNLESS the weak side wins ≥40%
        if weak_pct >= 0.40:
            pick = tid_weak
            pct = weak_pct * 100
        else:
            pick = tid_strong
            pct = winners.get(tid_strong, 0) / total * 100

        bracket[slot_name] = (pick, tid_to_name.get(pick, str(pick)), pct)
        advancing[slot_name] = pick

    return bracket


# ──────────────────────────────────────────────────────────────────────
#  Strategy 3: Upset Special — maximize differentiation
# ──────────────────────────────────────────────────────────────────────

def build_upset_special_bracket() -> dict[str, tuple[int, str, float]]:
    """Pick the lower seed whenever their simulated win rate significantly exceeds
    what their seed differential would normally predict (>25% sim win rate).
    Otherwise follow the modal pick."""
    bracket = {}
    advancing: dict[str, int] = dict(seed_to_team)

    for slot_info in slots:
        slot_name = slot_info["slot"]
        strong_key = slot_info["strong"]
        weak_key = slot_info["weak"]

        tid_strong = advancing.get(strong_key)
        tid_weak = advancing.get(weak_key)
        if tid_strong is None or tid_weak is None:
            continue

        winners = slot_winner_counts.get(slot_name, {})
        total = sum(winners.values()) or 1
        weak_pct = winners.get(tid_weak, 0) / total

        # Pick underdog when they have ≥25% win rate (value upset territory)
        if weak_pct >= 0.25:
            pick = tid_weak
            pct = weak_pct * 100
        else:
            pick = tid_strong
            pct = winners.get(tid_strong, 0) / total * 100

        bracket[slot_name] = (pick, tid_to_name.get(pick, str(pick)), pct)
        advancing[slot_name] = pick

    return bracket


# Build all three brackets
modal_bracket = build_modal_bracket()
chalk_bracket = build_chalk_plus_bracket()
upset_bracket = build_upset_special_bracket()


# ──────────────────────────────────────────────────────────────────────
#  Display Brackets Side by Side (key rounds)
# ──────────────────────────────────────────────────────────────────────

KEY_ROUNDS = {
    "Sweet 16": [s["slot"] for s in slots if s["slot"][1] == "3"],
    "Elite Eight": [s["slot"] for s in slots if s["slot"][1] == "4"],
    "Final Four": [s["slot"] for s in slots if s["slot"][1] == "5"],
    "Championship": [s["slot"] for s in slots if s["slot"][1] == "6"],
}

def fmt_pick(bracket, slot):
    if slot not in bracket:
        return "---"
    _, name, pct = bracket[slot]
    return f"{name} ({pct:.0f}%)"

for round_label, slot_list in KEY_ROUNDS.items():
    print(f"\n{'='*80}")
    print(f"  {round_label}")
    print(f"{'='*80}")
    print(f"{'Slot':<10} {'Modal':<28} {'Chalk+':<28} {'Upset Special':<28}")
    print("─" * 95)
    for sl in sorted(slot_list):
        m = fmt_pick(modal_bracket, sl)
        c = fmt_pick(chalk_bracket, sl)
        u = fmt_pick(upset_bracket, sl)
        print(f"{sl:<10} {m:<28} {c:<28} {u:<28}")


# Show how each bracket's champion search looks
print(f"\n{'='*80}")
print(f"  CHAMPION PICK")
print(f"{'='*80}")
for label, bracket in [("Modal", modal_bracket), ("Chalk+", chalk_bracket), ("Upset Special", upset_bracket)]:
    if "R6CH" in bracket:
        _, name, pct = bracket["R6CH"]
        print(f"  {label:<15} → {name} (won {pct:.1f}% of sims)")
    else:
        print(f"  {label:<15} → (no championship slot found)")

## 💡 Which Bracket to Submit?

**For a single bracket (or 1-3 entries):** Use the **Modal Bracket**. It maximizes expected correct picks by choosing the most likely winner at every decision point. This is the statistically optimal single bracket.

**For a medium-sized pool (5-20 entries):** Use **Chalk+**. It's close to optimal but sprinkles in just enough upsets to differentiate you from pure-chalk picks that half the pool will submit.

**For a large pool (50+ entries):** Consider the **Upset Special** alongside your Modal bracket. In large pools, everyone will have the chalk favorites — you need differentiated upsets to win. The heat-adjusted simulation naturally identifies teams with momentum that could bust brackets.

**Key insight from heat adjustments:** Teams with pre-tournament momentum (hot streaks) who then pull off a first-round upset get a compounding heat boost. This means the simulation naturally identifies "Cinderella-capable" teams better than static probabilities alone — watch for teams that show up in the Upset Special bracket but not Modal.

---

## Submission Ready Checks

Run the cell below to verify the current `artifacts/latest/submission.csv` is valid for upload and to preview confidence tails.

In [21]:
# Submission contract checks + confidence tails
sub = pl.read_csv(BASE / "submission.csv")

bad_id = sub.filter(~pl.col("ID").str.contains(r"^\d{4}_\d+_\d+$")).height
null_pred = sub.filter(pl.col("Pred").is_null()).height
out_of_range = sub.filter((pl.col("Pred") < 0.0) | (pl.col("Pred") > 1.0)).height

print("=== SUBMISSION CHECKS ===")
print(f"Rows          : {sub.height:,}")
print(f"Malformed IDs : {bad_id}")
print(f"Null Pred     : {null_pred}")
print(f"Out of range  : {out_of_range}")
print(f"Pred range    : [{sub['Pred'].min():.4f}, {sub['Pred'].max():.4f}]")
print(f"Pred mean     : {sub['Pred'].mean():.4f}")

print("\nMost confident TeamLow picks:")
print(sub.sort("Pred", descending=True).head(10))

print("\nMost confident TeamHigh picks:")
print(sub.sort("Pred", descending=False).head(10))

=== SUBMISSION CHECKS ===
Rows          : 132,133
Malformed IDs : 0
Null Pred     : 0
Out of range  : 0
Pred range    : [0.0100, 0.9900]
Pred mean     : 0.5079

Most confident TeamLow picks:
shape: (10, 2)
┌────────────────┬──────┐
│ ID             ┆ Pred │
│ ---            ┆ ---  │
│ str            ┆ f64  │
╞════════════════╪══════╡
│ 2026_1112_1202 ┆ 0.99 │
│ 2026_1112_1224 ┆ 0.99 │
│ 2026_1112_1225 ┆ 0.99 │
│ 2026_1112_1244 ┆ 0.99 │
│ 2026_1112_1250 ┆ 0.99 │
│ 2026_1112_1254 ┆ 0.99 │
│ 2026_1112_1295 ┆ 0.99 │
│ 2026_1112_1335 ┆ 0.99 │
│ 2026_1112_1341 ┆ 0.99 │
│ 2026_1112_1420 ┆ 0.99 │
└────────────────┴──────┘

Most confident TeamHigh picks:
shape: (10, 2)
┌────────────────┬──────┐
│ ID             ┆ Pred │
│ ---            ┆ ---  │
│ str            ┆ f64  │
╞════════════════╪══════╡
│ 2026_1202_1222 ┆ 0.01 │
│ 2026_1202_1235 ┆ 0.01 │
│ 2026_1202_1276 ┆ 0.01 │
│ 2026_1202_1345 ┆ 0.01 │
│ 2026_1224_1235 ┆ 0.01 │
│ 2026_1224_1276 ┆ 0.01 │
│ 2026_1224_1438 ┆ 0.01 │
│ 2026_1225_1235 ┆ 